In [1]:
%load_ext rpy2.ipython
sample = "SRR5279884"

In [2]:
%%bash -s "$sample"
set -euo pipefail
sample="$1"


trim_galore --illumina --stringency 10 --paired ${sample}_1.fastq.gz ${sample}_2.fastq.gz
cutadapt -j 0 -g ATGACGATAAGGCCGAGATGGCC -g GGAGAGGGTTAGGGATAGGCTTACCATTAATTAA -G ATGACGATAAGGCCGAGATGGCC -G GGAGAGGGTTAGGGATAGGCTTACCATTAATTAA --minimum-length 20 -O 10 --discard-untrimmed -o ${sample}_1_trim.fastq.gz -p ${sample}_2_trim.fastq.gz ${sample}_1_val_1.fq.gz ${sample}_2_val_2.fq.gz 


Trim Galore v2.6.0
unknown — linux/x86_64 — built 2026-11-19T00:09:00Z

Adapter: Illumina (AGATCGGAAGAGC)

Scanning for poly-G content...
Poly-G trimming: ENABLED (auto-detected). 1343 of 1000000 reads (0.13%) have poly-G tails (>=10bp) — consistent with 2-colour chemistry (NovaSeq/NextSeq). To disable: --no_poly_g
Trimming (paired-end):
  R1: SRR5279884_1.fastq.gz
  R2: SRR5279884_2.fastq.gz
  Output R1: SRR5279884_1_val_1.fq.gz
  Output R2: SRR5279884_2_val_2.fq.gz

=== Summary (Read 1) ===

Total reads processed:             21152974
Reads with adapters:                  70465 (0.3%)

=== Summary (Read 2) ===

Total reads processed:             21152974
Reads with adapters:                  75150 (0.4%)
R1 reads with poly-G trimmed:        845465 (4.0%)
  R1 poly-G bases removed:          6060312
R2 reads with poly-G trimmed:        944711 (4.5%)
  R2 poly-G bases removed:          9022277

=== Paired-end validation ===

Pairs analyzed:                    21152974
Pairs removed:   

This is cutadapt 5.2 with Python 3.12.16
Command line parameters: -g ATGACGATAAGGCCGAGATGGCC -g GGAGAGGGTTAGGGATAGGCTTACCATTAATTAA -G ATGACGATAAGGCCGAGATGGCC -G GGAGAGGGTTAGGGATAGGCTTACCATTAATTAA --minimum-length 20 -O 10 --discard-untrimmed -o SRR5279884_1_trim.fastq.gz -p SRR5279884_2_trim.fastq.gz SRR5279884_1_val_1.fq.gz SRR5279884_2_val_2.fq.gz
Processing paired-end reads on 1 core ...

=== Summary ===

Total read pairs processed:         20,962,993
  Read 1 with adapter:              20,790,501 (99.2%)
  Read 2 with adapter:              20,461,451 (97.6%)

== Read fate breakdown ==
Pairs that were too short:              42,404 (0.2%)
Pairs discarded as untrimmed:          659,489 (3.1%)
Pairs written (passing filters):    20,261,100 (96.7%)

Total basepairs processed: 6,240,744,484 bp
  Read 1: 3,123,576,810 bp
  Read 2: 3,117,167,674 bp
Total written (filtered):  4,678,440,244 bp (75.0%)
  Read 1: 2,339,591,571 bp
  Read 2: 2,338,848,673 bp

=== First read: Adapter 1 ===

Sequ

In [11]:
%%bash
set -euo pipefail

wget -c \
  https://ftp.ensembl.org/pub/release-84/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz

gzip -dc Homo_sapiens.GRCh38.cdna.all.fa.gz |
awk '
  /^>/ {
    keep = ($2 == "cdna:known" &&
            $0 ~ / gene_biotype:protein_coding( |$)/)
  }
  keep {print}
' > EnsemblBioMart_GRCh38_knowntranscripts.fa

# Check transcript count
grep -c '^>' EnsemblBioMart_GRCh38_knowntranscripts.fa

--2026-11-19 16:37:22--  https://ftp.ensembl.org/pub/release-84/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63432977 (60M) [application/x-gzip]
Saving to: ‘Homo_sapiens.GRCh38.cdna.all.fa.gz’

     0K .......... .......... .......... .......... ..........  0%  296K 3m29s
    50K .......... .......... .......... .......... ..........  0%  665K 2m31s
   100K .......... .......... .......... .......... ..........  0% 4.71M 1m45s
   150K .......... .......... .......... .......... ..........  0% 40.3M 79s
   200K .......... .......... .......... .......... ..........  0%  599K 84s
   250K .......... .......... .......... .......... ..........  0% 33.9M 70s
   300K .......... .......... .......... .......... ..........  0% 36.8M 60s
   350K .......... .......... .......... .......

156699


In [14]:
%%bash -s "$sample"
set -euo pipefail
sample="$1"

bash "$HOME/software/BBTools-indexfix/bbmap.sh" \
  -Xmx12g threads=4 \
  ref=EnsemblBioMart_GRCh38_knowntranscripts.fa \
  path=bbmap_index_fixed

java -ea   -Xmx12g -Xms12g -cp /home/imjeffli/software/BBTools-indexfix/current/ align2.BBMap build=1 overwrite=true fastareadlen=500 -Xmx12g threads=4 ref=EnsemblBioMart_GRCh38_knowntranscripts.fa path=bbmap_index_fixed
Executing align2.BBMap [build=1, overwrite=true, fastareadlen=500, -Xmx12g, threads=4, ref=EnsemblBioMart_GRCh38_knowntranscripts.fa, path=bbmap_index_fixed]
Version 40.05

Set threads to 4
No output file.
Writing reference.
Executing dna.FastaToChromArrays2 [EnsemblBioMart_GRCh38_knowntranscripts.fa, 1, writeinthread=false, genscaffoldinfo=true, retain, waitforwriting=false, gz=true, maxlen=536670912, writechroms=true, minscaf=1, midpad=300, startpad=8000, stoppad=8000, nodisk=false]

Set genScaffoldInfo=true
Writing chunk 1
Set genome to 1

Loaded Reference:	0.231 seconds.
Loading index for chunk 1-1, build 1
No index available; generating from reference genome: /orcd/scratch/orcd/008/imjeffli/secrify/bbmap_index_fixed/ref/index/1/chr1_index_k13_c2_b1.block
Indexing 

In [ ]:
%%bash -s "$sample"
set -euo pipefail
sample="$1"

bbmap.sh maxindel=300 xstag=us intronlen=10 ambig=best in=${sample}_1_trim.fastq.gz in2=${sample}_2_trim.fastq.gz out=${sample}_txmapped.sam -Xmx12g threads=4

java -ea   -Xmx12g -Xms12g -cp /home/imjeffli/.conda/envs/secrify/opt/bbmap-40.05-0/current/ align2.BBMap build=1 overwrite=true fastareadlen=500 maxindel=300 xstag=us intronlen=10 ambig=best in=SRR5279884_1_trim.fastq.gz in2=SRR5279884_2_trim.fastq.gz out=SRR5279884_txmapped.sam -Xmx12g threads=4
Executing align2.BBMap [build=1, overwrite=true, fastareadlen=500, maxindel=300, xstag=us, intronlen=10, ambig=best, in=SRR5279884_1_trim.fastq.gz, in2=SRR5279884_2_trim.fastq.gz, out=SRR5279884_txmapped.sam, -Xmx12g, threads=4]
Version 40.05

Set threads to 4
Retaining first best site only for ambiguous mappings.
Set genome to 1

Loaded Reference:	0.533 seconds.
Loading index for chunk 1-1, build 1
No index available; generating from reference genome: /orcd/scratch/orcd/008/imjeffli/secrify/ref/index/1/chr1_index_k13_c2_b1.block
Indexing threads started for block 0-1
Indexing threads finished for block 0-1
Generated Index:	6.454 seconds.
Finished Writing:	0.682 seconds.
Analyzed Index:   	2.

In [3]:
%%bash -s "$sample"
set -euo pipefail
sample="$1"

samtools view -bS ${sample}_txmapped.sam > ${sample}_txmapped.bam  
samtools flagstat ${sample}_txmapped.bam

samtools view -b -f2 ${sample}_txmapped.bam > ${sample}_pp_txmapped.bam 

samtools flagstat ${sample}_pp_txmapped.bam 
samtools sort -o ${sample}_pp_txmapped_sort.bam ${sample}_pp_txmapped.bam 
samtools flagstat ${sample}_pp_txmapped_sort.bam 
samtools index ${sample}_pp_txmapped_sort.bam 

40522199 + 1 in total (QC-passed reads + QC-failed reads)
40522199 + 1 primary
0 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
33488891 + 0 mapped (82.64% : 0.00%)
33488891 + 0 primary mapped (82.64% : 0.00%)
40522199 + 1 paired in sequencing
20261099 + 1 read1
20261100 + 0 read2
32453386 + 0 properly paired (80.09% : 0.00%)
32798808 + 0 with itself and mate mapped
690083 + 0 singletons (1.70% : 0.00%)
333518 + 0 with mate mapped to a different chr
110546 + 0 with mate mapped to a different chr (mapQ>=5)
32453386 + 0 in total (QC-passed reads + QC-failed reads)
32453386 + 0 primary
0 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
32453386 + 0 mapped (100.00% : N/A)
32453386 + 0 primary mapped (100.00% : N/A)
32453386 + 0 paired in sequencing
16226693 + 0 read1
16226693 + 0 read2
32453386 + 0 properly paired (100.00% : N/A)
32453386 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a differ

[bam_sort_core] merging from 12 files and 1 in-memory blocks...


32453386 + 0 in total (QC-passed reads + QC-failed reads)
32453386 + 0 primary
0 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
32453386 + 0 mapped (100.00% : N/A)
32453386 + 0 primary mapped (100.00% : N/A)
32453386 + 0 paired in sequencing
16226693 + 0 read1
16226693 + 0 read2
32453386 + 0 properly paired (100.00% : N/A)
32453386 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)
